# TrustGuard — prediction notebook (Track 2)
**This is the file uploaded to the portal.** It loads `model.pkl` and writes `submission.csv` (`id,prediction`).
No training cells — the platform sandbox-runs this on upload (one upload per 2h; a failed run burns the window).
The feature code below is inlined from `notebooks/submission.ipynb` §3–4 and must stay identical to it.
The model must be trained under the platform's versions: `pip install -r requirements-image.txt`.


In [1]:
import os
import numpy as np
import pandas as pd
import joblib

# --- feature engineering: keep byte-identical to the training notebook ---
MERCHANTS = ["cash_transfer", "electronics", "entertainment", "food_delivery", "gaming",
             "grocery", "luxury", "retail", "transport", "travel", "utilities"]
COUNTRIES = ["AU", "GB", "ID", "JP", "MY", "PH", "SG", "TH", "US", "VN"]
CHANNELS  = ["bank_transfer", "card_present", "ecommerce", "mobile_app"]
CAT_VOCAB = {"merchant_category": MERCHANTS, "country": COUNTRIES, "transaction_channel": CHANNELS}
FEATURES = ["transaction_amount", "transaction_hour", "merchant_category", "country",
            "transaction_channel", "transactions_last_24h", "spend_last_24h",
            "account_age", "new_device", "transactions_last_1h"]

def clean(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for c, vocab in CAT_VOCAB.items():
        out[c] = pd.Categorical(out[c], categories=vocab)
    return out

def make_features(df: pd.DataFrame) -> pd.DataFrame:
    out = clean(df)
    return out[FEATURES]


In [2]:
model, features = joblib.load("model.pkl")
input_path = os.environ.get("DATATHON_INPUT_PATH", "test.csv")
test_df = pd.read_csv(input_path)
preds = model.predict(make_features(test_df)[features])

sub = pd.DataFrame({"id": test_df["id"], "prediction": preds})
sub.to_csv("submission.csv", index=False)
assert list(sub.columns) == ["id", "prediction"] and len(sub) == len(test_df)
assert 0 <= sub.prediction.min() <= sub.prediction.max() <= 1
print(f"submission.csv written: {len(sub)} rows, mean p {sub.prediction.mean():.4f}")


submission.csv written: 12000 rows, mean p 0.0182
